# What the head is worth (2026-08-19)

The paper claims the model as a contribution and argues for two parts of the
head -- a four-band frequency split and a frequency-position encoding -- from
mechanism, on six folds. This measures them at sixteen.

Four arms, identical data, everything else fixed:

| arm | head | note |
|---|---|---|
| `freqpos` | the head as described | baseline, run first |
| `temporal` | frequency pooled away | the V7 path, largest difference |
| `freq` | band split, no position | separates the two components |
| `freqpos_noconfuser` | as described | minus 253 `Colobus_confuser` clips |

The existing sixteen-fold table is deliberately not reused as the baseline. It
was trained before nine labels were corrected on 2026-08-18, and an ablation
whose arms saw different data measures the data.

Each arm syncs to Drive as it finishes, so a disconnect costs the arm in
flight and nothing before it. Rerunning this notebook resumes.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection/upload_to_drive_CLEAN'
!mkdir -p /content/dataC && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataC/
!ls -la /content/dataC

In [ ]:
# The Drive copy carries labels from before 2026-08-18. Nine rows changed;
# the image pack did not. Patch the two small CSVs rather than re-upload 3 GB.
!python colab/fix_drive_labels.py

In [ ]:
A = '--manifest /content/dataC/manifest.csv --index /content/dataC/v13_index.csv --images /content/dataC/v13_images.npy --cache /content/dataC/v13_features.npy'
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/warm.csv --run-metadata /content/cacheC.run.json

In [ ]:
# Roughly ninety minutes per arm on a T4. Arms already in Drive are skipped.
!python colab/run_ablations.py

In [ ]:
# Read the arms back without retraining anything, once they exist.
import sys
sys.argv = ['x']
sys.path.insert(0, '/content/repo/colab')
import run_ablations
run_ablations.summarise([n for n, _ in run_ablations.ARMS])